In [2]:
!pip install folium

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import folium
import pandas as pd

# Ton dataframe avec colonnes 'lat', 'lon', 'nom'
import pyreadr

result = pyreadr.read_r("../data/processed/data_GPS.rds")
df = result[None]  # None = clé par défaut pour un objet RDS unique
# Créer la carte centrée sur la France

# Séparer la colonne en deux
df[['lat', 'lon']] = (
    df['Coordonnées.GPS.de.la.formation']
    .str.split(',', expand=True)
    .apply(lambda x: x.str.strip())
    .replace('', float('nan'))  # chaînes vides → NaN
    .astype(float)
)

# Voir combien de lignes sans coordonnées
print(f"Lignes sans coordonnées : {df['lat'].isna().sum()}")
print(df[['Coordonnées.GPS.de.la.formation', 'lat', 'lon']].head())
print(df[['Coordonnées.GPS.de.la.formation', 'lat', 'lon']].head())

# Option A — supprimer les lignes sans coordonnées
df_carte = df.dropna(subset=['lat', 'lon'])

# Option B — les garder dans le df mais les exclure de la carte
# (utiliser df_carte = df.dropna(subset=['lat','lon']) juste pour folium/contextily)


Lignes sans coordonnées : 38
  Coordonnées.GPS.de.la.formation       lat      lon
0               45.74148, 4.88109  45.74148  4.88109
1                48.9458, 2.36341  48.94580  2.36341
2                46.0446, 4.07402  46.04460  4.07402
3                45.7818, 4.86821  45.78180  4.86821
4                45.74149, 4.8807  45.74149  4.88070
  Coordonnées.GPS.de.la.formation       lat      lon
0               45.74148, 4.88109  45.74148  4.88109
1                48.9458, 2.36341  48.94580  2.36341
2                46.0446, 4.07402  46.04460  4.07402
3                45.7818, 4.86821  45.78180  4.86821
4                45.74149, 4.8807  45.74149  4.88070


In [4]:
m = folium.Map(location=[46.5, 2.5], zoom_start=6)

# Ajouter les marqueurs
for _, row in df_carte.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        #popup=row['nom'],
        color='steelblue',
        fill=True
    ).add_to(m)

m.save("carte.html")  # Ouvrir dans un navigateur

In [5]:
# Habillages disponibles nativement dans Folium :
tiles_options = [
    "OpenStreetMap",          # classique
    "CartoDB positron",       # fond clair minimaliste ⭐
    "CartoDB dark_matter",    # fond sombre ⭐
    "Stamen Terrain",         # relief
    "Stamen Toner",           # noir & blanc
    "Stamen Watercolor",      # aquarelle stylisée
]

# Exemple avec CartoDB Positron
m = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB positron")

In [6]:
m = folium.Map(location=[46.5, 2.5], zoom_start=6)

# Ajouter plusieurs fonds de carte
folium.TileLayer("CartoDB positron", name="Clair").add_to(m)
folium.TileLayer("CartoDB dark_matter", name="Sombre").add_to(m)
folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
folium.TileLayer("Stamen Terrain", name="Relief").add_to(m)

# Ajouter tes marqueurs...
for _, row in df_carte.iterrows():
    folium.CircleMarker([row['lat'], row['lon']], radius=5,
                        #popup=row['nom'], 
                        fill=True).add_to(m)

# Contrôleur de couches (le sélecteur visuel)
folium.LayerControl().add_to(m)

m.save("carte_multi_habillages.html")

In [9]:
import branca.colormap as cm
var_habillage = 'taux_acces_num'  # remplace par ta variable d'intérêt
# Créer une palette de couleurs liée à ta variable
colormap = cm.LinearColormap(
    colors=['red', 'yellow', 'green'],
    vmin=df_carte[var_habillage].min(),
    vmax=df_carte[var_habillage].max(),
    caption=var_habillage  # légende
)

In [25]:
!pip install branca

Defaulting to user installation because normal site-packages is not writeable


In [12]:
# Affichage inline dans le notebook
m = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB positron")

# Ajouter plusieurs fonds de carte
folium.TileLayer("CartoDB positron", name="Clair").add_to(m)
folium.TileLayer("CartoDB dark_matter", name="Sombre").add_to(m)
folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
folium.TileLayer("Stamen Terrain", name="Relief").add_to(m)

for _, row in df_carte.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color=colormap(row[var_habillage]),  # couleur selon la variable
        fill=True,
        fill_color=colormap(row[var_habillage]),
        fill_opacity=0.8,
        popup=folium.Popup(row['Établissement'], max_width=300)
        # popup=folium.Popup(
        #     f"<b>{row[var_habillage]}</b>",  # remplace par d'autres colonnes si besoin
        #     max_width=200
        # )
    ).add_to(m)

colormap.add_to(m)


# Contrôleur de couches (le sélecteur visuel)
folium.LayerControl().add_to(m)


m.save("carte_multi_habillages.html")
#display(m)  # ou juste m en dernière ligne de cellule